# Prompt Chaining: Outline → Blog Post

This is the simplest kind of LangGraph workflow: two steps that run one after
another, where the output of the first step feeds into the second.

1. **create_outline** — ask the model for an outline on the given topic.
2. **create_blog** — ask the model to turn that outline into a full blog post.

No branching, no loops — just `START → create_outline → create_blog → END`.

This pattern is called **prompt chaining**: instead of asking one model call
to do everything at once ("write me a blog post"), you break the task into
smaller steps and feed each step's output into the next. Smaller, focused
prompts are usually more reliable than one giant prompt trying to do
everything at once.


In [28]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv


**Imports**
- `StateGraph, START, END` — the pieces used to build the graph: `StateGraph` builds it, `START`/`END` mark where it begins and finishes.
- `ChatGoogleGenerativeAI` — wrapper for calling Google's Gemini model.
- `TypedDict` — used below to describe the shape of the data (`BlogState`) that flows through the graph.
- `load_dotenv` — reads your `.env` file so the Gemini API key is available without hardcoding it in the notebook.


In [29]:
load_dotenv()  # Load environment variables from .env file

MODEL_NAME = "gemini-3.5-flash"
precise_model = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0,
    max_output_tokens=512,
    thinking_budget=0,
)

**Setup**
- `load_dotenv()` loads your API key (and anything else) from a `.env` file so the model can authenticate.
- `precise_model` is the Gemini model used for both steps. `temperature=0` makes replies consistent/deterministic — good here since we don't want the outline changing shape every run. `thinking_budget=0` turns off the model's internal "thinking" step, which this simple task doesn't need and which would otherwise eat into `max_output_tokens` before any visible text is written.


In [30]:
class BlogState(TypedDict):
    topic: str
    outline: str
    content: str

**State (`BlogState`)**

Every LangGraph node reads from and writes to the same dictionary-like
object, shaped by this `TypedDict`. Think of it as a shared clipboard that
gets passed from node to node:
- `topic` — set once at the start; the blog post's subject.
- `outline` — filled in by `create_outline`.
- `content` — filled in by `create_blog`, using `outline`.


In [31]:
def create_outline(state: BlogState) -> BlogState:
    prompt = f"Create a detailed outline for a blog post on the topic: '{state['topic']}'."
    outline = precise_model.invoke(prompt).content
    state['outline'] = outline
    return state

**Node 1: `create_outline`**

Takes `state['topic']`, asks the model to write a detailed outline for it,
and saves the result into `state['outline']`. `.invoke(prompt)` sends the
prompt to the model and returns a message object; `.content` pulls out just
the text of the reply.


In [32]:
def create_blog(state: BlogState) -> BlogState:
    title = state["topic"]
    outline = state["outline"]

    prompt = f"""
Write a blog post titled "{title}" based on the following outline:

{outline}
"""

    response = precise_model.invoke(prompt)

    state["content"] = response.content

    return state

**Node 2: `create_blog`**

Takes the `topic` and the `outline` produced by the previous node, and asks
the model to write the full blog post from them. The result goes into
`state['content']` — this is the value we'll print once the graph finishes.


In [33]:
graph=StateGraph(BlogState)

# node
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

# edge
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)

workflow= graph.compile()


**Wiring the graph**

1. `graph = StateGraph(BlogState)` creates the graph and tells it what shape
   the state is.
2. `add_node(name, function)` registers each step under a name.
3. `add_edge(...)` connects the steps in order: `START → create_outline →
   create_blog → END`.
4. `graph.compile()` turns this definition into a runnable `workflow`.

Since there's no branching or looping here, every edge is a plain
`add_edge` (no `add_conditional_edges`) — the graph always takes the exact
same path, in the exact same order, every time it runs.


**Running it**

`initial_state` only needs `topic` — LangGraph fills in `outline` and
`content` as the nodes run. `workflow.invoke(...)` executes `create_outline`
then `create_blog` in order and returns the completed state dictionary as
`final_state`. We print `final_state["content"]` to see the finished blog
post the two-step chain produced.


In [34]:
initial_state = {"topic": "The Future of Artificial Intelligence"}
final_state = workflow.invoke(initial_state)
print(final_state["content"])


[{'type': 'text', 'text': '# The Future of Artificial Intelligence: Beyond the Horizon of Innovation\n\nWe are living in the prologue of the artificial intelligence revolution. What happens when the main story begins? \n\nOnly a few years ago, AI was a concept relegated to science fiction novels and high-tech research labs. Today, it is woven into our daily routines. We use generative AI tools like ChatGPT to draft emails, Midjourney to create stunning digital art, and predictive algorithms to navigate morning traffic. Yet, the rapid advancements we are witnessing today are merely the opening act. \n\nAs we look toward the next decade, a profound question emerges: *How will AI evolve, and what does that mean for humanity?* \n\nThe future of AI is not just about building smarter machines; it is about the deep integration of AI into the very fabric of human society. This evolution will reshape global industries, redefine our relationship with technology, and challenge our ethical framewo